# Bloch-Sphere Playground

The Bloch sphere is the standard mental model for a single qubit. By the end of this notebook you will see *why* it is useful, and you will be able to predict measurement outcomes by looking at a point on the sphere.

**Objectives:**
- Parametrize any single-qubit pure state with two angles `theta` and `phi`
- Predict P(|0>) and P(|1>) from the angles alone
- Use an interactive widget to drag the state around and watch probabilities update
- See how gates act geometrically (rotations around the sphere)

**Reference:** See [`../GUIDE.md`](../GUIDE.md).

<!-- browser-runnable -->


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrow
np.set_printoptions(precision=3, suppress=True)

## 1. The Bloch parametrization

Any single-qubit pure state can be written, up to an irrelevant global phase, as

$$|\psi\rangle = \cos(\theta/2) |0\rangle + e^{i \phi} \sin(\theta/2) |1\rangle$$

where

- $\theta \in [0, \pi]$ is the **polar angle** (north pole = $|0\rangle$, south pole = $|1\rangle$)
- $\phi \in [0, 2\pi)$ is the **azimuthal angle** (relative phase between the two amplitudes)

Two angles. The whole single-qubit state space lives on a sphere.

In [ ]:
def state_from_bloch(theta, phi):
    """Build |psi> = cos(theta/2)|0> + e^{i phi} sin(theta/2)|1>."""
    return np.array([
        np.cos(theta / 2),
        np.exp(1j * phi) * np.sin(theta / 2),
    ], dtype=complex)

# Famous states
for name, (theta, phi) in [
    ('|0>',  (0,         0)),
    ('|1>',  (np.pi,     0)),
    ('|+>',  (np.pi/2,   0)),
    ('|->',  (np.pi/2,   np.pi)),
    ('|+i>', (np.pi/2,   np.pi/2)),
    ('|-i>', (np.pi/2, 3*np.pi/2)),
]:
    psi = state_from_bloch(theta, phi)
    print(f'{name:>5}  theta={theta:.3f}  phi={phi:.3f}   psi={psi}')

## 2. Measurement probabilities from `theta`

Because the amplitudes are $\cos(\theta/2)$ and $e^{i\phi}\sin(\theta/2)$, the Born rule gives

$$P(0) = \cos^2(\theta/2), \qquad P(1) = \sin^2(\theta/2).$$

The azimuthal angle `phi` does **not** affect computational-basis measurement. It carries the relative phase, which only matters when you apply a gate before measuring.

In [ ]:
thetas = np.linspace(0, np.pi, 200)
p0 = np.cos(thetas / 2) ** 2
p1 = np.sin(thetas / 2) ** 2

plt.plot(thetas, p0, label='P(0) = cos^2(theta/2)', color='#ff9900')
plt.plot(thetas, p1, label='P(1) = sin^2(theta/2)', color='#146eb4')
plt.axvline(np.pi/2, color='gray', linestyle='--', label='equator (theta = pi/2)')
plt.xlabel('theta')
plt.ylabel('probability')
plt.title('Measurement probabilities vs. Bloch polar angle')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. Draw the Bloch circle

We will draw the X-Z slice of the Bloch sphere (the page of the screen). The full 3D sphere adds the Y axis to capture `phi`, but the slice is enough to build intuition.

In [ ]:
def plot_bloch_slice(theta, phi=0, title=None, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))
    circle = Circle((0, 0), 1, fill=False, color='black', linewidth=1.5)
    ax.add_patch(circle)
    # Project onto the X-Z plane:  x = sin(theta) cos(phi), z = cos(theta)
    x = np.sin(theta) * np.cos(phi)
    z = np.cos(theta)
    ax.annotate(
        '', xy=(x*0.92, z*0.92), xytext=(0, 0),
        arrowprops=dict(arrowstyle='->', color='#ff9900', lw=2),
    )
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.axvline(0, color='gray', lw=0.5, ls='--')
    ax.text(0,  1.15, '|0>', ha='center')
    ax.text(0, -1.15, '|1>', ha='center')
    ax.text( 1.15, 0, '|+>', va='center')
    ax.text(-1.15, 0, '|->', va='center')
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
plot_bloch_slice(0,         0,        title='|0>',  ax=axes[0])
plot_bloch_slice(np.pi,     0,        title='|1>',  ax=axes[1])
plot_bloch_slice(np.pi/2,   0,        title='|+>',  ax=axes[2])
plot_bloch_slice(np.pi/2,   np.pi,    title='|->',  ax=axes[3])
plt.tight_layout()
plt.show()

## 4. Interactive playground

Drag the sliders. Watch the arrow move and the probability bars update. This cell uses `ipywidgets` — install with `pip install ipywidgets` if needed.

(If widgets aren't available, the next cell does the same thing with a static grid.)

In [ ]:
try:
    from ipywidgets import interact, FloatSlider

    def explore(theta=np.pi/4, phi=0.0):
        psi = state_from_bloch(theta, phi)
        probs = np.abs(psi) ** 2

        fig, (ax_bloch, ax_bar) = plt.subplots(1, 2, figsize=(9, 3.5))
        plot_bloch_slice(theta, phi, ax=ax_bloch, title=f'theta={theta:.2f}, phi={phi:.2f}')

        ax_bar.bar(['|0>', '|1>'], probs, color=['#ff9900', '#146eb4'])
        ax_bar.set_ylim(0, 1)
        ax_bar.set_title('measurement probabilities')
        for i, p in enumerate(probs):
            ax_bar.text(i, p + 0.02, f'{p:.3f}', ha='center')
        plt.tight_layout()
        plt.show()

    interact(
        explore,
        theta=FloatSlider(min=0, max=np.pi,   step=0.05, value=np.pi/4),
        phi  =FloatSlider(min=0, max=2*np.pi, step=0.05, value=0.0),
    )
except ImportError:
    print('ipywidgets not installed — see the static grid below instead.')

In [ ]:
# Static fallback: a grid of states across the Bloch slice.
thetas = [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]
fig, axes = plt.subplots(1, len(thetas), figsize=(15, 3.5))
for ax, theta in zip(axes, thetas):
    psi = state_from_bloch(theta, 0)
    probs = np.abs(psi) ** 2
    plot_bloch_slice(theta, 0, ax=ax,
                     title=f'theta={theta:.2f}\nP(0)={probs[0]:.2f}, P(1)={probs[1]:.2f}')
plt.tight_layout()
plt.show()

## 5. Gates as rotations

Every single-qubit gate is a **rotation** of the Bloch sphere. For example:

- $X$ rotates 180° about the X-axis — sends $|0\rangle$ to $|1\rangle$
- $Z$ rotates 180° about the Z-axis — sends $|+\rangle$ to $|-\rangle$
- $H$ rotates 180° about the $(X+Z)/\sqrt{2}$ axis — sends $|0\rangle$ to $|+\rangle$
- $R_y(\alpha)$ rotates by $\alpha$ about the Y-axis — useful for state preparation

Watch what happens when we apply gates and re-extract the Bloch angles.

In [ ]:
def bloch_from_state(psi):
    """Recover (theta, phi) from a state vector (up to global phase)."""
    a, b = psi
    # Strip a global phase so a is real and non-negative.
    if abs(a) > 1e-12:
        phase = a / abs(a)
        a, b = a / phase, b / phase
    theta = 2 * np.arctan2(abs(b), abs(a))
    phi = np.angle(b) if abs(b) > 1e-12 else 0.0
    return theta, phi

H = (1/np.sqrt(2)) * np.array([[1,  1], [1, -1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)

zero = np.array([1, 0], dtype=complex)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, (name, psi) in zip(axes, [('|0>', zero), ('X|0>', X @ zero), ('H|0>', H @ zero)]):
    theta, phi = bloch_from_state(psi)
    plot_bloch_slice(theta, phi, ax=ax, title=f'{name}\ntheta={theta:.2f}, phi={phi:.2f}')
plt.tight_layout()
plt.show()

## 6. Exercises

These three exercises are the gate. Each has tiered hints -- expand only
what you need -- and a check cell that tells you when you have it. If you
can clear them without consulting the cells above, you are ready for
`01-foundations`.

### Exercise 1 — Locate |+i> on the sphere

Section 1 listed $|+i\rangle = (|0\rangle + i|1\rangle)/\sqrt{2}$ among the
famous states. Work out its Bloch angles and say where it lives on the sphere.

Define `iplus_theta` and `iplus_phi`: the polar and azimuthal angles of
$|+i\rangle$.

<details><summary>Hint 1 — nudge</summary>

Match the two amplitudes of $|+i\rangle$ against the parametrization
$\cos(\theta/2)|0\rangle + e^{i\phi}\sin(\theta/2)|1\rangle$. Equal
magnitudes on $|0\rangle$ and $|1\rangle$ pin down $\theta$; the factor
multiplying $|1\rangle$ pins down $\phi$.

</details>
<details><summary>Hint 2 — approach</summary>

Build $|+i\rangle$ as a length-2 complex NumPy array and pass it to
`bloch_from_state(...)` from Section 5 — it returns `(theta, phi)` directly.
Unpack the pair into the two names the check expects.

</details>

In [ ]:
# Exercise 1: Find the Bloch angles of |+i> = (|0> + i|1>)/sqrt(2).
# Define: iplus_theta, iplus_phi -- the polar and azimuthal angles of |+i>.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert 0 <= iplus_theta <= np.pi, "theta is the polar angle -- it lives in [0, pi]"
    assert abs(np.cos(iplus_theta)) < 1e-6, (
        "|+i> weights |0> and |1> equally, so it should sit on the equator"
    )
    assert (
        abs(np.vdot(np.array([1, 1j]) / np.sqrt(2), state_from_bloch(iplus_theta, iplus_phi)))
        > 0.999
    ), "state_from_bloch(iplus_theta, iplus_phi) should rebuild |+i> up to a global phase"

### Exercise 2 — Probability from the polar angle

A qubit sits at $\theta = \pi/3$ (any $\phi$). Predict $P(0)$ in your head
first — then confirm with code.

Define `p0_pi3`: the probability of measuring $|0\rangle$ for this state.

<details><summary>Hint 1 — nudge</summary>

Section 2's Born-rule formula gives the computational-basis probabilities from
$\theta$ alone — $\phi$ never enters. Mind the half-angle.

</details>
<details><summary>Hint 2 — approach</summary>

Evaluate the Section 2 expression for $P(0)$ at $\theta = \pi/3$ with
`np.cos` — or build the state with `state_from_bloch(...)` and take the
squared magnitude of its $|0\rangle$ amplitude.

</details>

In [ ]:
# Exercise 2: Predict P(0) for a qubit at theta = pi/3, then confirm with code.
# Define: p0_pi3 -- the probability of measuring |0>.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert 0.0 <= p0_pi3 <= 1.0, "a probability lives in [0, 1]"
    assert p0_pi3 > 0.5, (
        "theta = pi/3 is above the equator -- the state leans toward |0>"
    )
    assert abs(p0_pi3 - abs(state_from_bloch(np.pi / 3, 0.0)[0]) ** 2) < 1e-9, (
        "P(0) should match the Born rule for the |0> amplitude at theta = pi/3"
    )

### Exercise 3 — Hadamard on |1>

Section 5 showed $H$ sending $|0\rangle$ to the equator. Now apply $H$ to
$|1\rangle$ and locate the result on the sphere.

Define `h1_theta` and `h1_phi`: the Bloch angles of $H|1\rangle$, recovered
with `bloch_from_state`.

<details><summary>Hint 1 — nudge</summary>

Every single-qubit gate is a rotation, and $H$ swaps the roles of the Z and X
axes — so a pole must land somewhere on the equator. Which equatorial point is
the question.

</details>
<details><summary>Hint 2 — approach</summary>

Build $|1\rangle$ as a length-2 complex array, apply the `H` matrix from
Section 5 with `@`, and pass the product to `bloch_from_state(...)` — the
same pattern Section 5 used for $H|0\rangle$.

</details>

In [ ]:
# Exercise 3: Apply H to |1> and locate the result on the Bloch sphere.
# Define: h1_theta, h1_phi -- the Bloch angles of H|1>.

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    assert 0 <= h1_theta <= np.pi, "theta is the polar angle -- it lives in [0, pi]"
    assert abs(np.cos(h1_theta)) < 1e-6, (
        "H sends a pole to the equator -- P(0) and P(1) should come out equal"
    )
    assert (
        abs(np.vdot(H @ np.array([0, 1], dtype=complex), state_from_bloch(h1_theta, h1_phi)))
        > 0.999
    ), "state_from_bloch(h1_theta, h1_phi) should rebuild H|1> up to a global phase"

### Solutions

In [ ]:
# --- Exercise 1 ---
iplus_psi = np.array([1, 1j]) / np.sqrt(2)
iplus_theta, iplus_phi = bloch_from_state(iplus_psi)
print(iplus_theta, iplus_phi)           # (pi/2, pi/2)
# Lives on the equator, +Y direction.

# --- Exercise 2 ---
p0_pi3 = np.cos((np.pi / 3) / 2) ** 2
print('P(0) =', p0_pi3)                 # cos^2(pi/6) = 3/4

# --- Exercise 3 ---
one = np.array([0, 1], dtype=complex)
h1_theta, h1_phi = bloch_from_state(H @ one)
print(h1_theta, h1_phi)                 # (pi/2, pi)  -- that's |->

## You finished the prerequisites module.

If the self-checks across all six notebooks felt routine, head to [`../../01-foundations/GUIDE.md`](../../01-foundations/GUIDE.md).

If three or more questions tripped you up, replay the corresponding notebook before continuing. The foundations module assumes everything here is automatic.